# Sequence Encoder Comparison: Lorenz Attractor from Partial Observations

This notebook compares several sequence-to-next-step architectures for
reconstructing the Lorenz attractor from **partial observations** (only
the $x$ coordinate).

Instead of requiring a hand-tuned delay embedding, each architecture
takes a raw $(T, D)$ window and encodes it into an $N$-dimensional latent
space, which is then decoded to predict the **next time step**.

**Architectures tested:**
1. Transformer (no positional encoding)
2. Transformer (with positional encoding)
3. State Space Model / S4 (no positional encoding)
4. State Space Model / S4 (with positional encoding)
5. Temporal Convolutional Network (TCN)
6. TCN + Spatial Convolutions

All models use **FNN regularization** on the $N$-dimensional latent.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import time
from collections import OrderedDict

from JacobianODE.dysts_sim.flows import Lorenz
from JacobianODE.fnn import (
    TransformerEmbedding,
    SSMEmbedding,
    TCNEmbedding,
    TCNSpatialEmbedding,
    FNN,
    compute_variances,
    compute_s_dim,
)

plt.rcParams.update({
    'figure.figsize': (12, 4),
    'font.size': 11,
})

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

## 1. Generate Lorenz Data

In [ ]:
# Generate Lorenz trajectories
lorenz = Lorenz()
data_dict = lorenz.make_trajectory(
    n_periods=12,
    pts_per_period=100,
    num_ics=16,
    new_ic_mode='random',
    traj_offset_sd=0.2,
    return_times=True,
)

full_data = data_dict['values']  # (num_ics, T, 3)
dt = data_dict['dt']
print(f'Full data shape: {full_data.shape}, dt = {dt:.4f}')

# Add small observation noise
obs_noise = 0.01
np.random.seed(42)
full_data_noisy = full_data + obs_noise * np.random.randn(*full_data.shape)

# Partial observation: only x coordinate (D=1)
partial_data = full_data_noisy[:, :, [0]]  # (num_ics, T, 1)
print(f'Partial observation shape: {partial_data.shape}')

# Train / test split by initial conditions
n_train = 12
train_data = partial_data[:n_train]
test_data = partial_data[n_train:]
train_full = full_data_noisy[:n_train]
test_full = full_data_noisy[n_train:]
print(f'Train: {train_data.shape}, Test: {test_data.shape}')

In [ ]:
# Visualize the Lorenz attractor and partial observation
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 3D projection (x vs z)
for i in range(min(4, n_train)):
    axes[0].plot(train_full[i, :, 0], train_full[i, :, 2], alpha=0.5, lw=0.5)
axes[0].set_xlabel('x'); axes[0].set_ylabel('z')
axes[0].set_title('Lorenz attractor (x vs z)')

# Partial observation time series
for i in range(min(4, n_train)):
    axes[1].plot(train_data[i, :200, 0], alpha=0.7, lw=0.8)
axes[1].set_xlabel('Time step'); axes[1].set_ylabel('x')
axes[1].set_title('Partial observation (x only, first 200 steps)')

# x-y phase space
for i in range(min(4, n_train)):
    axes[2].plot(train_full[i, :, 0], train_full[i, :, 1], alpha=0.5, lw=0.5)
axes[2].set_xlabel('x'); axes[2].set_ylabel('y')
axes[2].set_title('Lorenz attractor (x vs y)')

plt.tight_layout()
plt.show()

## 2. Define Architectures and Hyperparameters

In [ ]:
# Shared hyperparameters
N_LATENT = 6          # Latent dimension (Lorenz has 3 dims, but we give extra capacity)
TIME_WINDOW = 64      # Input sequence length
N_FEATURES = 1        # Partial observation dimension (x only)
FNN_STRENGTH = 0.01   # FNN regularization strength
TRAIN_STEPS = 200     # Training epochs
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
SEED = 42

fnn_reg = FNN(FNN_STRENGTH)

# Define all model configurations
model_configs = OrderedDict([
    ('Transformer (no PE)', {
        'class': TransformerEmbedding,
        'kwargs': dict(
            use_positional_encoding=False,
            d_model=64, n_heads=4, n_layers=3,
            dim_feedforward=128, dropout=0.1,
        ),
    }),
    ('Transformer (PE)', {
        'class': TransformerEmbedding,
        'kwargs': dict(
            use_positional_encoding=True,
            d_model=64, n_heads=4, n_layers=3,
            dim_feedforward=128, dropout=0.1,
        ),
    }),
    ('SSM (no PE)', {
        'class': SSMEmbedding,
        'kwargs': dict(
            use_positional_encoding=False,
            d_model=64, d_state=64, n_layers=3, dropout=0.1,
        ),
    }),
    ('SSM (PE)', {
        'class': SSMEmbedding,
        'kwargs': dict(
            use_positional_encoding=True,
            d_model=64, d_state=64, n_layers=3, dropout=0.1,
        ),
    }),
    ('TCN', {
        'class': TCNEmbedding,
        'kwargs': dict(
            n_channels=64, kernel_size=7, n_layers=4, dropout=0.1,
        ),
    }),
    ('TCN + Spatial', {
        'class': TCNSpatialEmbedding,
        'kwargs': dict(
            n_channels=64, kernel_size_temporal=7,
            kernel_size_spatial=3, n_layers=4, dropout=0.1,
        ),
    }),
])

print(f'{len(model_configs)} architectures to compare')

## 3. Train All Models

In [ ]:
results = OrderedDict()

for name, cfg in model_configs.items():
    print(f'\n{"="*60}')
    print(f'Training: {name}')
    print(f'{"="*60}')

    model = cfg['class'](
        n_latent=N_LATENT,
        time_window=TIME_WINDOW,
        n_features=N_FEATURES,
        random_state=SEED,
        latent_regularizer=FNN(FNN_STRENGTH),
        **cfg['kwargs'],
    )

    t0 = time.time()
    model.fit(
        train_data,
        train_steps=TRAIN_STEPS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        verbose=1,
        optimizer='adamw',
    )
    train_time = time.time() - t0

    # Evaluate on test set
    preds = model.predict_next(test_data)     # (n_windows, 1)
    latent = model.transform(test_data)       # (n_windows, N_LATENT)

    # Compute ground truth next steps for test data
    from JacobianODE.fnn.sequence_models import sliding_windows
    test_std = model._standardize(test_data)
    _, y_true = sliding_windows(test_std, TIME_WINDOW)

    mse = float(np.mean((preds - y_true) ** 2))

    # Normalized variances of latent
    norm_var = compute_variances(latent, normalize=True)

    results[name] = {
        'model': model,
        'mse': mse,
        'norm_var': norm_var,
        'latent': latent,
        'preds': preds,
        'y_true': y_true,
        'train_time': train_time,
        'n_params': model.count_parameters(),
        'history': model.train_history,
    }

    print(f'  Test MSE: {mse:.6f}')
    print(f'  Params:   {model.count_parameters():,}')
    print(f'  Time:     {train_time:.1f}s')

print(f'\nAll {len(results)} models trained.')

## 4. Results Comparison

In [ ]:
# Print summary table
print(f'{"Model":25s} {"Test MSE":>12s} {"Params":>10s} {"Train (s)":>10s}')
print('-' * 60)
for name, r in results.items():
    print(f'{name:25s} {r["mse"]:12.6f} {r["n_params"]:10,} {r["train_time"]:10.1f}')

# Identify best model
best_name = min(results, key=lambda k: results[k]['mse'])
print(f'\nBest model: {best_name} (MSE = {results[best_name]["mse"]:.6f})')

In [ ]:
# Bar chart of test MSE
names = list(results.keys())
mses = [results[n]['mse'] for n in names]
colors = ['#2ecc71' if n == best_name else '#3498db' for n in names]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(names)), mses, color=colors, edgecolor='white', linewidth=0.8)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=30, ha='right')
ax.set_ylabel('Test MSE (next-step prediction)')
ax.set_title('Architecture Comparison: Next-Step Prediction on Lorenz (x only)')

for bar, mse_val in zip(bars, mses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'{mse_val:.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 5. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, r in results.items():
    axes[0].plot(r['history']['pred_loss'], label=name, alpha=0.8)
    axes[1].plot(r['history']['reg_loss'], label=name, alpha=0.8)

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Prediction Loss (MSE)')
axes[0].set_title('Prediction Loss'); axes[0].legend(fontsize=8)
axes[0].set_yscale('log')

axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('FNN Regularization Loss')
axes[1].set_title('FNN Regularization Loss'); axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 6. Latent Space Analysis

In [ ]:
# Normalized variance spectrum of latent dimensions
fig, ax = plt.subplots(figsize=(10, 5))

for name, r in results.items():
    ax.plot(range(1, len(r['norm_var'])+1), r['norm_var'], 'o-', label=name, alpha=0.8)

ax.set_xlabel('Latent Dimension')
ax.set_ylabel('Normalized Variance')
ax.set_title('Latent Variance Spectrum (FNN-regularized)')
ax.legend(fontsize=8)
ax.set_yscale('log')
ax.set_xticks(range(1, N_LATENT+1))
plt.tight_layout()
plt.show()

In [ ]:
# 2D latent projections (first two principal latent dims)
from sklearn.decomposition import PCA

n_models = len(results)
fig, axes = plt.subplots(1, n_models, figsize=(4*n_models, 4))
if n_models == 1:
    axes = [axes]

for ax, (name, r) in zip(axes, results.items()):
    lat = r['latent']
    pca = PCA(n_components=2)
    lat_2d = pca.fit_transform(lat)
    ax.scatter(lat_2d[:, 0], lat_2d[:, 1], s=0.5, alpha=0.3, c='tab:blue')
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

fig.suptitle('Latent Space (PCA projection)', y=1.02)
plt.tight_layout()
plt.show()

## 7. Prediction Quality

In [ ]:
# Predicted vs true next step
n_models = len(results)
fig, axes = plt.subplots(1, n_models, figsize=(4*n_models, 4))
if n_models == 1:
    axes = [axes]

for ax, (name, r) in zip(axes, results.items()):
    # Subsample for clarity
    n_plot = min(2000, len(r['preds']))
    idx = np.random.choice(len(r['preds']), n_plot, replace=False)
    ax.scatter(r['y_true'][idx, 0], r['preds'][idx, 0], s=1, alpha=0.3)
    lims = [r['y_true'][:, 0].min(), r['y_true'][:, 0].max()]
    ax.plot(lims, lims, 'r--', lw=0.8, alpha=0.5)
    ax.set_title(f'{name}\nMSE={r["mse"]:.4f}', fontsize=9)
    ax.set_xlabel('True'); ax.set_ylabel('Predicted')
    ax.set_aspect('equal')

fig.suptitle('Next-Step Prediction: True vs Predicted', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Time series comparison for one test trajectory
fig, axes = plt.subplots(len(results), 1, figsize=(14, 3*len(results)), sharex=True)

# Use first test trajectory
test_single = test_data[0]  # (T, 1)
test_std_single = results[list(results.keys())[0]]['model']._standardize(test_single)
t_plot = 200  # show first 200 steps

for ax, (name, r) in zip(axes, results.items()):
    model = r['model']
    preds_single = model.predict_next(test_single)
    test_std_local = model._standardize(test_single)

    # Ground truth (shifted by window)
    true_vals = test_std_local[TIME_WINDOW:TIME_WINDOW+t_plot, 0]
    pred_vals = preds_single[:t_plot, 0]

    ax.plot(true_vals, 'k-', lw=1, label='True', alpha=0.8)
    ax.plot(pred_vals, '--', lw=1, label='Predicted', alpha=0.8)
    ax.set_ylabel('x (std)')
    ax.set_title(name, fontsize=10)
    ax.legend(loc='upper right', fontsize=8)

axes[-1].set_xlabel('Time step')
fig.suptitle('Next-Step Prediction on Test Trajectory', y=1.01)
plt.tight_layout()
plt.show()

## 8. Autoregressive Trajectory Prediction

In [ ]:
# Autoregressive rollout from seed
N_ROLLOUT = 300  # steps to predict forward
test_traj = test_data[0]  # (T, 1)

fig, axes = plt.subplots(len(results), 1, figsize=(14, 3*len(results)), sharex=True)

for ax, (name, r) in zip(axes, results.items()):
    model = r['model']
    # Seed with first TIME_WINDOW points, predict forward
    seed = test_traj[:TIME_WINDOW + 50]  # give a bit extra for context
    pred_traj = model.predict_trajectory(seed, n_steps=N_ROLLOUT)

    # Ground truth (un-standardized)
    true_traj = test_traj[TIME_WINDOW + 50 : TIME_WINDOW + 50 + N_ROLLOUT, 0]
    n_compare = min(len(true_traj), len(pred_traj))

    ax.plot(true_traj[:n_compare], 'k-', lw=1, label='True', alpha=0.8)
    ax.plot(pred_traj[:n_compare, 0], '--', lw=1, label='Autoregressive', alpha=0.8)
    ax.set_ylabel('x')
    ax.set_title(name, fontsize=10)
    ax.legend(loc='upper right', fontsize=8)

axes[-1].set_xlabel('Time step')
fig.suptitle(f'Autoregressive Rollout ({N_ROLLOUT} steps)', y=1.01)
plt.tight_layout()
plt.show()

## 9. FNN Regularization Sweep (Best Architecture)

Sweep the FNN regularization strength on the best-performing architecture
to find the optimal trade-off between prediction accuracy and latent
structure.

In [ ]:
# FNN sweep on the best architecture
best_cfg = model_configs[best_name]
lambda_values = [0, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]

sweep_results = {'lambda': [], 'mse': [], 'norm_var': []}

for lam in lambda_values:
    print(f'  lambda={lam} ...', end=' ')
    reg = FNN(lam) if lam > 0 else None
    model = best_cfg['class'](
        n_latent=N_LATENT,
        time_window=TIME_WINDOW,
        n_features=N_FEATURES,
        random_state=SEED,
        latent_regularizer=reg,
        **best_cfg['kwargs'],
    )
    model.fit(train_data, train_steps=TRAIN_STEPS, batch_size=BATCH_SIZE,
              learning_rate=LEARNING_RATE, verbose=0, optimizer='adamw')

    preds = model.predict_next(test_data)
    test_std = model._standardize(test_data)
    _, y_true = sliding_windows(test_std, TIME_WINDOW)
    mse = float(np.mean((preds - y_true) ** 2))

    latent = model.transform(test_data)
    norm_var = compute_variances(latent, normalize=True)

    sweep_results['lambda'].append(lam)
    sweep_results['mse'].append(mse)
    sweep_results['norm_var'].append(norm_var)
    print(f'MSE={mse:.6f}')

print('Sweep complete.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MSE vs lambda
x_labels = [str(v) for v in sweep_results['lambda']]
axes[0].plot(x_labels, sweep_results['mse'], 'o-', color='tab:blue')
axes[0].set_xlabel(r'FNN $\lambda$')
axes[0].set_ylabel('Test MSE')
axes[0].set_title(f'FNN Sweep: {best_name}')
axes[0].tick_params(axis='x', rotation=45)

# Variance spectrum vs lambda
for lam, nv in zip(sweep_results['lambda'], sweep_results['norm_var']):
    axes[1].plot(range(1, len(nv)+1), nv, 'o-', label=f'$\\lambda$={lam}', alpha=0.8)
axes[1].set_xlabel('Latent Dimension')
axes[1].set_ylabel('Normalized Variance')
axes[1].set_title('Latent Variance Spectrum')
axes[1].set_yscale('log')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 10. Partial Observations with D=2

Repeat the comparison observing both $x$ and $y$ to see how extra
information affects each architecture.

In [ ]:
# Partial observation: (x, y) -> D=2
partial_data_2d = full_data_noisy[:, :, :2]  # (num_ics, T, 2)
train_data_2d = partial_data_2d[:n_train]
test_data_2d = partial_data_2d[n_train:]

results_2d = OrderedDict()

for name, cfg in model_configs.items():
    print(f'Training: {name} (D=2) ...', end=' ')
    model = cfg['class'](
        n_latent=N_LATENT,
        time_window=TIME_WINDOW,
        n_features=2,
        random_state=SEED,
        latent_regularizer=FNN(FNN_STRENGTH),
        **cfg['kwargs'],
    )
    model.fit(train_data_2d, train_steps=TRAIN_STEPS, batch_size=BATCH_SIZE,
              learning_rate=LEARNING_RATE, verbose=0, optimizer='adamw')

    preds = model.predict_next(test_data_2d)
    test_std = model._standardize(test_data_2d)
    _, y_true = sliding_windows(test_std, TIME_WINDOW)
    mse = float(np.mean((preds - y_true) ** 2))

    results_2d[name] = {'mse': mse, 'n_params': model.count_parameters()}
    print(f'MSE={mse:.6f}')

print()
# Comparison table: D=1 vs D=2
print(f'{"Model":25s} {"MSE (D=1)":>12s} {"MSE (D=2)":>12s}')
print('-' * 52)
for name in results.keys():
    print(f'{name:25s} {results[name]["mse"]:12.6f} {results_2d[name]["mse"]:12.6f}')

In [ ]:
# Side-by-side bar chart: D=1 vs D=2
names = list(results.keys())
mse_1d = [results[n]['mse'] for n in names]
mse_2d = [results_2d[n]['mse'] for n in names]

x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width/2, mse_1d, width, label='D=1 (x only)', color='#3498db')
ax.bar(x + width/2, mse_2d, width, label='D=2 (x, y)', color='#e74c3c')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=30, ha='right')
ax.set_ylabel('Test MSE')
ax.set_title('Architecture Comparison: D=1 vs D=2 Partial Observations')
ax.legend()
plt.tight_layout()
plt.show()

## 11. Summary

In [ ]:
print('='*70)
print('FINAL SUMMARY')
print('='*70)
print(f'\nTask: Next-step prediction of Lorenz system from partial observations')
print(f'Latent dim: {N_LATENT}, Window: {TIME_WINDOW}, FNN lambda: {FNN_STRENGTH}')
print(f'Training: {TRAIN_STEPS} epochs, lr={LEARNING_RATE}\n')

print(f'{"Model":25s} {"MSE (D=1)":>12s} {"MSE (D=2)":>12s} {"Params":>10s} {"Time (s)":>10s}')
print('-' * 72)
for name in results.keys():
    print(f'{name:25s} {results[name]["mse"]:12.6f} {results_2d[name]["mse"]:12.6f} '
          f'{results[name]["n_params"]:10,} {results[name]["train_time"]:10.1f}')

best_1d = min(results, key=lambda k: results[k]['mse'])
best_2d = min(results_2d, key=lambda k: results_2d[k]['mse'])
print(f'\nBest (D=1): {best_1d}  (MSE={results[best_1d]["mse"]:.6f})')
print(f'Best (D=2): {best_2d}  (MSE={results_2d[best_2d]["mse"]:.6f})')